# 🎯 Skillnox AI — Fine-Tune Qwen3-8B V2 (Production)

**Model**: Qwen3-8B (text-only, latest gen)  
**Method**: QLoRA (4-bit) via Unsloth  
**Dataset**: 50,000 examples (Resume, Answer Eval, Questions, GD, Aptitude, Communication)  
**GPU**: Kaggle P100/T4 (16GB VRAM)  
**Training time**: ~8-10 hours (5 epochs)  

## Features
- ✅ **Checkpoint Resume** — If session stops, restart and continue from last checkpoint
- ✅ **Validation Split** — Monitors eval loss to catch overfitting
- ✅ **Loss Visualization** — Plots training/eval loss curves
- ✅ **Comprehensive Testing** — 8 test cases covering all Skillnox tasks
- ✅ **GGUF Export** — Ready for Ollama deployment
- ✅ **HuggingFace Push** — Optional auto-upload

## Setup Checklist
1. Accelerator: **GPU P100** (or T4)
2. Internet: **ON**
3. Add your `skillnox-training-data` dataset as input
4. (Optional) Add Kaggle secret `HF_TOKEN` for HuggingFace push

---
## Cell 1 — Install Dependencies & GPU Check

In [ ]:
%%time
!pip install -q -U unsloth transformers datasets peft accelerate bitsandbytes trl safetensors huggingface_hub matplotlib hf_transfer

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"

import torch

print("\n" + "="*60)
print("ENVIRONMENT CHECK")
print("="*60)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
else:
    print("⚠️  NO GPU DETECTED — Training will be extremely slow!")
    print("   Go to Settings → Accelerator → GPU P100")

print(f"\nDisk space: {os.popen('df -h /kaggle/working').read()}")
print("=== Dependencies installed! ===")

---
## Cell 2 — Configuration Dashboard
**Edit these values to control training behavior.**

In [ ]:
# =============================================================
# TRAINING CONFIGURATION - Edit these values
# =============================================================

# Model
MODEL_NAME = "unsloth/Qwen3-8B-bnb-4bit"  # Pre-quantized Qwen3-8B
MAX_SEQ_LEN = 512                          # Max tokens per example (512 is safe and fast)

# LoRA Configuration
LORA_RANK = 16             # LoRA rank (16 is very stable and memory-efficient)
LORA_ALPHA = 32            # Usually 2x rank
LORA_DROPOUT = 0.05

# Training
NUM_EPOCHS = 5             # More epochs for better learning
BATCH_SIZE = 2             # Per-device batch size
GRAD_ACCUM_STEPS = 4       # Effective batch = 2 * 4 = 8
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01

# Checkpointing
SAVE_STEPS = 500           # Save checkpoint every N steps
SAVE_TOTAL_LIMIT = 3       # Keep last N checkpoints
EVAL_STEPS = 500           # Evaluate every N steps
LOGGING_STEPS = 25         # Log metrics every N steps

# Dataset
MAX_EXAMPLES = 15000       # Set to 15000 to prevent system RAM OOM on Kaggle (13GB limit)
VALIDATION_SPLIT = 0.05    # 5% held out for validation

# Paths
OUTPUT_DIR = "/kaggle/working/output"
LORA_DIR = "/kaggle/working/lora_adapter"
GGUF_DIR = "/kaggle/working/skillnox-gguf"

# Resume from checkpoint
RESUME_FROM_CHECKPOINT = True  # Auto-detect and resume if checkpoint exists

# Estimate training time
# ~1.2 seconds per step on P100 with batch_size=2, seq_len=1024
ESTIMATED_SECS_PER_STEP = 1.2

print("="*60)
print("TRAINING CONFIGURATION")
print("="*60)
print(f"Model:          {MODEL_NAME}")
print(f"LoRA Rank:      {LORA_RANK} (alpha: {LORA_ALPHA})")
print(f"Epochs:         {NUM_EPOCHS}")
print(f"Batch (eff.):   {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Learning Rate:  {LEARNING_RATE}")
print(f"Sequence Len:   {MAX_SEQ_LEN}")
print(f"Checkpoints:    Every {SAVE_STEPS} steps (keep {SAVE_TOTAL_LIMIT})")
print(f"Eval:           Every {EVAL_STEPS} steps")
print(f"Max Examples:   {MAX_EXAMPLES}")
print(f"Resume:         {RESUME_FROM_CHECKPOINT}")
print("="*60)


---
## Cell 3 — Load & Prepare Dataset

In [ ]:
%%time
import json
import random
from datasets import Dataset

# --- Find the dataset ---
DATA_PATH = None

# Check Kaggle input first
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".jsonl"):
            candidate = os.path.join(root, f)
            # Prefer the extended dataset
            if "extended" in f:
                DATA_PATH = candidate
                break
            elif DATA_PATH is None:
                DATA_PATH = candidate
    if DATA_PATH and "extended" in DATA_PATH:
        break

# If not found, download
if DATA_PATH is None:
    print("Dataset not found in /kaggle/input, downloading...")
    !kaggle datasets download surendravarikallu1/skillnox-training-data -p /kaggle/working/data --unzip
    for f in os.listdir("/kaggle/working/data"):
        if f.endswith(".jsonl"):
            DATA_PATH = f"/kaggle/working/data/{f}"
            break

print(f"Using dataset: {DATA_PATH}")

# --- Load raw data ---
raw_data = []
with open(DATA_PATH) as f:
    for line in f:
        raw_data.append(json.loads(line))
print(f"\nLoaded {len(raw_data):,} total examples")

# --- Limit examples if configured ---
random.seed(42)
random.shuffle(raw_data)
if MAX_EXAMPLES and MAX_EXAMPLES < len(raw_data):
    raw_data = raw_data[:MAX_EXAMPLES]
    print(f"Limited to {MAX_EXAMPLES:,} examples")

# --- Format into ChatML ---
SYSTEM_PROMPT = (
    "You are SkillnoxAI, an expert AI-powered interview preparation and placement assistant. "
    "Your capabilities include: Interview Question Generation, Answer Evaluation (score 0-100), "
    "Resume Analysis, Communication Assessment, Group Discussion Evaluation, and Aptitude Test Evaluation. "
    "Be STRICT but constructive. Do NOT inflate scores. "
    "For evaluations, always use: Score: [number]\nFeedback: [text]. "
    "Focus on Indian campus placement context (TCS, Infosys, Wipro, Accenture, etc.)."
)

def format_example(example):
    instruction = example["instruction"]
    inp = json.dumps(example["input"]) if isinstance(example["input"], dict) else str(example["input"])
    out = json.dumps(example["output"]) if isinstance(example["output"], dict) else str(example["output"])
    
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{instruction}\n{inp}<|im_end|>\n"
        f"<|im_start|>assistant\n{out}<|im_end|>"
    )
    return {"text": text}

formatted = [format_example(ex) for ex in raw_data]

# --- Split into train/validation ---
split_idx = int(len(formatted) * (1 - VALIDATION_SPLIT))
train_formatted = formatted[:split_idx]
eval_formatted = formatted[split_idx:]

train_dataset = Dataset.from_list(train_formatted)
eval_dataset = Dataset.from_list(eval_formatted)

# --- Show distribution ---
types = {}
for ex in raw_data:
    t = ex["instruction"].split()[0]
    types[t] = types.get(t, 0) + 1

print(f"\n{'='*40}")
print(f"DATASET SUMMARY")
print(f"{'='*40}")
print(f"Training:   {len(train_dataset):,} examples")
print(f"Validation: {len(eval_dataset):,} examples")
print(f"\nDistribution:")
for t, c in sorted(types.items(), key=lambda x: -x[1]):
    print(f"  {t}: {c:,}")

# Estimate training time
est_steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM_STEPS)
est_total_steps = est_steps_per_epoch * NUM_EPOCHS
est_hours = (est_total_steps * ESTIMATED_SECS_PER_STEP) / 3600
print(f"\n⏱️  Estimated steps/epoch: {est_steps_per_epoch:,}")
print(f"⏱️  Estimated total steps: {est_total_steps:,}")
print(f"⏱️  Estimated training time: {est_hours:.1f} hours")
if est_hours > 11:
    print(f"\n⚠️  WARNING: Training may exceed 12-hour Kaggle limit!")
    print(f"   Consider reducing NUM_EPOCHS to {max(1, int(11 / (est_hours / NUM_EPOCHS)))}")
    print(f"   Or set MAX_EXAMPLES to reduce dataset size")

print(f"\nSample (first 300 chars):")
print(formatted[0]['text'][:300] + "...")
# --- Memory Cleanup to prevent Kaggle CPU OOM ---
import gc
del raw_data
del formatted
del train_formatted
del eval_formatted
gc.collect()
print("Free CPU memory cleaned up successfully!")


---
## Cell 4 — Load Qwen3-8B with QLoRA

In [ ]:
%%time
from unsloth import FastLanguageModel

print(f"Loading {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,  # auto-detect
)

# Add LoRA adapters with higher rank for better capacity
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # 60% less VRAM
    random_state=42,
)

print("\n=== Model loaded with LoRA adapters ===")
model.print_trainable_parameters()

# Show VRAM usage
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / (1024**3)
    reserved = torch.cuda.memory_reserved() / (1024**3)
    print(f"\nVRAM allocated: {allocated:.2f} GB")
    print(f"VRAM reserved:  {reserved:.2f} GB")

---
## Cell 5 — Train with Checkpoints & Resume Support

**If training gets interrupted:**
1. Re-run Cell 1 → Cell 4 (they're fast)
2. Run this cell — it will automatically find the last checkpoint and resume
3. Training continues from where it stopped

In [ ]:
%%time
import glob
import time
import traceback

# --- Check for existing checkpoint ---
resume_checkpoint = None
if RESUME_FROM_CHECKPOINT:
    checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
    if checkpoints:
        resume_checkpoint = checkpoints[-1]
        step_num = resume_checkpoint.split("-")[-1]
        print(f"[RESUME] Resuming from checkpoint: {resume_checkpoint}")
        print(f"   (Step {step_num})")
    else:
        print("[NEW] No checkpoint found - starting fresh training")

# --- Configure trainer (compatible with latest TRL) ---
try:
    from trl import SFTTrainer, SFTConfig
    print("Using SFTConfig (TRL >= 0.14)")
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=SFTConfig(
            output_dir=OUTPUT_DIR,
            dataset_text_field="text",
            max_seq_length=MAX_SEQ_LEN,
            packing=False,
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM_STEPS,
            learning_rate=LEARNING_RATE,
            lr_scheduler_type="cosine",
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            fp16=True,
            logging_steps=LOGGING_STEPS,
            save_steps=SAVE_STEPS,
            save_total_limit=SAVE_TOTAL_LIMIT,
            eval_strategy="steps",
            eval_steps=EVAL_STEPS,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            optim="adamw_8bit",
            seed=42,
            report_to="none",
        ),
    )
except ImportError:
    from trl import SFTTrainer
    from transformers import TrainingArguments
    print("Using TrainingArguments (older TRL)")
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        packing=False,
        args=TrainingArguments(
            output_dir=OUTPUT_DIR,
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM_STEPS,
            learning_rate=LEARNING_RATE,
            lr_scheduler_type="cosine",
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            fp16=True,
            logging_steps=LOGGING_STEPS,
            save_steps=SAVE_STEPS,
            save_total_limit=SAVE_TOTAL_LIMIT,
            eval_strategy="steps",
            eval_steps=EVAL_STEPS,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            optim="adamw_8bit",
            seed=42,
            report_to="none",
        ),
    )

print("Trainer created successfully!")

# --- Start training ---
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)
print(f"Epochs:          {NUM_EPOCHS}")
print(f"Train examples:  {len(train_dataset)}")
print(f"Eval examples:   {len(eval_dataset)}")
print(f"Batch (eff.):    {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"LR:              {LEARNING_RATE}")
print(f"Optimizer:       AdamW 8-bit")
print(f"Checkpoints:     {OUTPUT_DIR}")
if resume_checkpoint:
    print(f"Resuming from:   {resume_checkpoint}")
print("="*60)

start_time = time.time()
stats = trainer.train(resume_from_checkpoint=resume_checkpoint)
elapsed = time.time() - start_time
hours, remainder = divmod(elapsed, 3600)
minutes, seconds = divmod(remainder, 60)

print(f"\nTRAINING COMPLETE!")
print(f"Total steps:    {stats.global_step}")
print(f"Final loss:     {stats.training_loss:.4f}")
print(f"Training time:  {int(hours)}h {int(minutes)}m {int(seconds)}s")

# Save final LoRA adapter
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"\nLoRA adapter saved to {LORA_DIR}")

# Save training stats
import json
stats_dict = {
    "total_steps": stats.global_step,
    "final_loss": stats.training_loss,
    "training_hours": round(elapsed / 3600, 2),
    "epochs": NUM_EPOCHS,
    "dataset_size": len(train_dataset),
    "lora_rank": LORA_RANK,
}
with open("/kaggle/working/training_stats.json", "w") as f:
    json.dump(stats_dict, f, indent=2)
print("Training stats saved.")

---
## Cell 6 — Training Loss Visualization

In [ ]:
import matplotlib.pyplot as plt

if "trainer" not in dir() or trainer is None:
    print("WARNING: trainer not available. Skipping loss visualization.")
else:
    train_losses = []
    eval_losses = []
    train_steps = []
    eval_steps_log = []

    for log in trainer.state.log_history:
        if "loss" in log and "eval_loss" not in log:
            train_losses.append(log["loss"])
            train_steps.append(log["step"])
        if "eval_loss" in log:
            eval_losses.append(log["eval_loss"])
            eval_steps_log.append(log["step"])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    ax1.plot(train_steps, train_losses, "b-", alpha=0.7, linewidth=1)
    ax1.set_xlabel("Steps")
    ax1.set_ylabel("Training Loss")
    ax1.set_title("Training Loss Over Time")
    ax1.grid(True, alpha=0.3)

    if eval_losses:
        ax2.plot(eval_steps_log, eval_losses, "r-o", markersize=4)
        ax2.set_xlabel("Steps")
        ax2.set_ylabel("Eval Loss")
        ax2.set_title("Validation Loss Over Time")
        ax2.grid(True, alpha=0.3)
    else:
        ax2.text(0.5, 0.5, "No eval data", ha="center", va="center", fontsize=14)
        ax2.set_title("Validation Loss")

    plt.tight_layout()
    plt.savefig("/kaggle/working/training_loss.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Final train loss: {train_losses[-1]:.4f}")
    if eval_losses:
        print(f"Final eval loss:  {eval_losses[-1]:.4f}")
        print(f"Best eval loss:   {min(eval_losses):.4f}")


---
## Cell 7 — Comprehensive Model Testing

In [ ]:
FastLanguageModel.for_inference(model)

def test_model(system, user_msg, max_tokens=300):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_msg},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to("cuda")
    out = model.generate(inputs, max_new_tokens=max_tokens, temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

SYS = "You are SkillnoxAI, an expert interview preparation and placement assistant."

TESTS = [
    ("Technical Question Generation",
     '{"question_type": "technical", "context": "Python backend development"}'),
    ("HR Question Generation",
     '{"question_type": "hr", "context": "TCS campus placement"}'),
    ("Answer Evaluation",
     '{"question": "What is OOP?", "answer": "OOP stands for Object Oriented Programming. It has classes and objects."}'),
    ("Strong Answer Evaluation",
     '{"question": "Explain SOLID principles.", "answer": "SOLID consists of 5 principles: Single Responsibility (a class should have one reason to change), Open/Closed (open for extension, closed for modification), Liskov Substitution (subtypes should be substitutable for base types), Interface Segregation (prefer small, specific interfaces), and Dependency Inversion (depend on abstractions not concretions). At my previous company, we refactored our payment service using these principles which reduced bug count by 40%."}'),
    ("Resume Analysis",
     '{"resume_text": "Amit Kumar\\nBackend Developer\\nEducation: B.Tech CSE from NIT Trichy (CGPA: 8.5)\\nSkills: Python, Django, PostgreSQL, Docker, AWS\\nExperience: 2 years\\nProjects: REST API gateway, Auth microservice"}'),
    ("Communication Assessment",
     '{"question": "Tell me about yourself", "answer": "Well, uh, I am a developer. I do coding and stuff. I like Python I guess."}'),
    ("Aptitude Evaluation",
     '{"question": "If 6 workers can complete a task in 12 days, how many days will 8 workers take?", "answer": "Total work = 6 x 12 = 72 worker-days. With 8 workers: 72/8 = 9 days.", "difficulty": "easy"}'),
    ("Group Discussion Evaluation",
     '{"topic": "Is AI a threat to jobs?", "response": "I think AI will take all jobs. Everything will be automated."}'),
]

print("="*60)
print("MODEL TEST RESULTS")
print("="*60)

for i, (name, prompt) in enumerate(TESTS, 1):
    print(f"\n{'─'*60}")
    print(f"TEST {i}: {name}")
    print(f"{'─'*60}")
    result = test_model(SYS, prompt)
    print(f"Input:  {prompt[:100]}..." if len(prompt) > 100 else f"Input:  {prompt}")
    print(f"Output: {result}")

print(f"\n{'='*60}")
print("All tests complete!")
print(f"{'='*60}")

---
## Cell 8 — Export to GGUF (for Ollama deployment)
**This takes ~10-15 minutes.**

In [ ]:
%%time
print("Exporting to GGUF (q4_k_m quantization)...")
print("This takes ~10-15 minutes.\n")

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)

print("\n=== GGUF Export Complete ===")
for f in os.listdir(GGUF_DIR):
    fpath = os.path.join(GGUF_DIR, f)
    size_gb = os.path.getsize(fpath) / (1024**3)
    print(f"  {f}: {size_gb:.2f} GB")

print("\n📥 Download from the Output tab on the right sidebar.")

---
## Cell 9 — (Optional) Push to HuggingFace Hub
**Requires HF_TOKEN Kaggle secret.**

In [ ]:
from huggingface_hub import HfApi, login

import os
HF_TOKEN = os.environ.get("HF_TOKEN", "")

try:
    if HF_TOKEN and HF_TOKEN.startswith("hf_"):
        if HF_TOKEN:
            login(token=HF_TOKEN)
        
        # Push LoRA adapter
        repo_id = "suren3101/skillnox-qwen3-8b-lora"
        print(f"Pushing LoRA adapter to {repo_id}...")
        model.push_to_hub(repo_id, token=HF_TOKEN)
        tokenizer.push_to_hub(repo_id, token=HF_TOKEN)
        print(f"[OK] LoRA adapter pushed to https://huggingface.co/{repo_id}")
        
        # Push GGUF
        gguf_repo = "suren3101/skillnox-qwen3-8b-gguf"
        api = HfApi()
        import os
        for f in os.listdir(GGUF_DIR):
            if f.endswith(".gguf"):
                print(f"Uploading {f} to {gguf_repo}...")
                api.upload_file(
                    path_or_fileobj=os.path.join(GGUF_DIR, f),
                    path_in_repo=f,
                    repo_id=gguf_repo,
                    repo_type="model",
                    token=HF_TOKEN if HF_TOKEN else None,
                )
                print(f"[OK] GGUF uploaded to https://huggingface.co/{gguf_repo}")
    else:
        print("[WARN] HF_TOKEN is empty or invalid. Skipping push.")
except Exception as e:
    print(f"[WARN] HuggingFace push failed: {e}")


---
## Cell 10 — Backup & Deployment Instructions

In [ ]:
import shutil

# Zip the LoRA adapter for backup
shutil.make_archive("/kaggle/working/skillnox-lora-adapter", "zip", LORA_DIR)
size_mb = os.path.getsize("/kaggle/working/skillnox-lora-adapter.zip") / (1024**2)
print(f"💾 LoRA adapter backup: skillnox-lora-adapter.zip ({size_mb:.1f} MB)")

# List all output files
print(f"\n{'='*60}")
print("OUTPUT FILES")
print(f"{'='*60}")
for f in sorted(os.listdir("/kaggle/working")):
    fpath = f"/kaggle/working/{f}"
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        if size > 1024**3:
            print(f"  {f}: {size/(1024**3):.2f} GB")
        elif size > 1024**2:
            print(f"  {f}: {size/(1024**2):.1f} MB")
        else:
            print(f"  {f}: {size/1024:.1f} KB")

print(f"\n{'='*60}")
print("🚀 DEPLOYMENT INSTRUCTIONS")
print(f"{'='*60}")
print("""
1. Download the .gguf file from the Output tab (right sidebar)

2. On your local machine:
   cd skillnox_ai/python-ai/models/
   # Place the .gguf file here

3. Update Modelfile (line 1):
   FROM ./unsloth.Q4_K_M.gguf

4. Create Ollama model:
   ollama create skillnox-qwen:latest -f models/Modelfile

5. Test:
   ollama run skillnox-qwen:latest

6. Start the service:
   start_service.bat
""")

# Load and display training stats
if os.path.exists("/kaggle/working/training_stats.json"):
    with open("/kaggle/working/training_stats.json") as f:
        final_stats = json.load(f)
    print(f"\n📊 TRAINING SUMMARY")
    print(f"   Steps:        {final_stats['total_steps']}")
    print(f"   Final loss:   {final_stats['final_loss']:.4f}")
    print(f"   Training hrs: {final_stats['training_hours']}")
    print(f"   Dataset size: {final_stats['dataset_size']:,}")
    print(f"   LoRA rank:    {final_stats['lora_rank']}")